In [143]:
import jax
jax.config.update("jax_enable_x64", True)   # before any array creation
import jax.numpy as jnp

import jaxquantum as jqt
from gkp_optimal_control.gate_optimization import (
    optimize_gate_sequence, GateBounds, OptimizerConfig,
)
from gkp_optimal_control.plotting import (
    plot_photon_number,
    plot_wigner,
    set_plot_style,
)
from gkp_optimal_control.states import (
    cat_states,
    fock_state,
    gkp_states,
    squeezed_vacuum,
)
from gkp_optimal_control.utils import wigner_trajectory

In [4]:
n_fock = 100
gkp_delta = 0.3
gkp_cutoff = 10
gkp_alpha = jnp.sqrt(jnp.pi / 2)
gkp_beta = jnp.sqrt(jnp.pi / 2) * 1j

gkp_0, gkp_1 = gkp_states(n_fock, gkp_alpha, gkp_beta, gkp_delta, gkp_cutoff)
fock_1 = jqt.basis(n_fock, 1)
fock_3 = jqt.basis(n_fock, 3)
vac = jqt.basis(n_fock, 0)
def _unit(v):
    return v / jnp.linalg.norm(v)


gkp_x_plus = _unit(gkp_0 + gkp_1)

psi_i = vac
psi_f = fock_3

from gkp_optimal_control.gate_results import gate_result_path, load_gate_result, save_gate_result
_path = gate_result_path('data/gate_results/gate_optimization_test', 'ecd', 5)
if _path.exists():
    res = load_gate_result(_path)
    print("loaded cached result from " + str(_path))
else:
    res = optimize_gate_sequence('ecd', 5, psi_i, psi_f, disp_method='quadrature', loss_type='log_infidelity', bounds=GateBounds(max_disp=4.0, n_leak=8), optimizer=OptimizerConfig(n_seeds=16, n_adam_iters=2000, peak_lr=0.03))
    save_gate_result(res, _path, initial_state=psi_i, overwrite=True, source_notebook='gate_optimization_test')


print(res.summary())

NameError: name 'jnp' is not defined

In [5]:
from gkp_optimal_control.gate_optimization import sequence_history
import numpy as np, jaxquantum as jqt, matplotlib.pyplot as plt

pair, bound, grid = 0, 6.0, 120
hist = sequence_history(res, psi_i)
idxs = [0] + list(range(3, 2 * res.n_gates + 2, 2))   # input, then one per layer

def to_array(frame):
    f = np.asarray(frame)
    if f.ndim == 2:                      # ECD block state: trace out the qubit
        rho = np.outer(f[0], f[0].conj()) + np.outer(f[1], f[1].conj())
        return rho / np.real(np.trace(rho))
    return (f / np.linalg.norm(f)).reshape(-1, 1)

fig, axes = plt.subplots(1, len(idxs), figsize=(3.0 * len(idxs), 3.2))
for ax, i in zip(np.atleast_1d(axes), idxs):
    label = "input" if i == 0 else f"layer {i // 2}"
    plot_wigner(state=to_array(hist[i][pair]), x_bound=bound, y_bound=bound,
                ax=ax, title=label, grid_points=grid)
fig.tight_layout()

NameError: name 'res' is not defined

In [171]:
print(res.params)

{'thetas': array([1.57079633, 1.8257233 , 0.86817495, 1.49794609, 1.57079633]), 'phis': array([ 1.27886525, -2.84966158, -3.43352373,  3.43352373,  1.27886525]), 'betas': array([-0.49865353+2.02107188j, -1.5324474 +0.38426535j,
        0.89563848+1.13775618j, -0.26027532+0.45281912j]), 'disp_magnitudes': array([2.08167886, 1.57989079, 1.44798391, 0.52229149]), 'per_pair_fidelity': array([0.9487439])}
